# 06 · Outputs & audit — the paper trail

Everything one happy-path run writes, and who reads it.

## Setup — the lab bench

In [1]:
import json
import sys
from pathlib import Path

# Work from the repo root no matter where the kernel was started.
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "notebooks" / "pipeline_lab.py").exists(), (
    f"llm-mailroom repo root not found above {Path.cwd()}"
)
sys.path.insert(0, str(ROOT / "notebooks"))

import pipeline_lab as lab


## What you'll see

- the four artifact surfaces: manifests, SQLite catalog, archive bin, audit chain
- how `doc_id` / `trace_id` / `matter_id` thread through all of them
- which fields The-Mailroom visualizer consumes

**Honesty label:** all artifacts are real files written by the real pipeline into the sandbox base dir.

## One happy run

In [2]:
env6 = lab.open_sandbox()
r = lab.run_document(env6, lab.DOC_CONTRACT,
                     classification=lab.CLASSIFY_CONTRACT_HIGH,
                     extraction=lab.EXTRACT_HIGH,
                     filename="papertrail.txt")
f = r["final"]
print("doc_id:   ", f["doc_id"])
print("trace_id: ", f.get("trace_id"))
print("matter_id:", f["matter_id"])


doc_id:    06bc81f0-d2ca-4d7f-b433-83dda94f924a
trace_id:  
matter_id: LAB-MATTER


## Everything on disk

In [3]:
arts = lab.artifacts(Path(env6["base_dir"]))
print(sorted(arts.keys()))


['archive_bin', 'audit_chain', 'catalog', 'failed_bin', 'manifests', 'review_bin']


## The manifest (what the visualizer's conveyor reads)

In [4]:
name, man = next(iter(arts["manifests"].items()))
print(name)
for k in sorted(man):
    v = man[k]
    s = json.dumps(v) if isinstance(v, (dict, list)) else str(v)
    print(f"  {k:26s} {s[:70]}")


06bc81f0-d2ca-4d7f-b433-83dda94f924a.json
  classification_attempts    1
  classification_confidence  0.98
  contract_subtype           other
  created_at                 2026-08-24T23:17:17.221715Z
  doc_id                     06bc81f0-d2ca-4d7f-b433-83dda94f924a
  doc_type                   contract
  escalation_reason          None
  extracted_data             {"parties": ["Acme Corp", "Beta LLC"], "effective_date": "2024-01-01",
  extraction_attempts        1
  extraction_confidence      0.96
  matter_id                  LAB-MATTER
  original_filename          papertrail.txt
  review_decision            None
  stage                      archived
  trace_id                   
  updated_at                 2026-08-24T23:17:17.221716Z


## The catalog row

In [5]:
rows = arts["catalog"]
for row in rows:
    print(row)


tables
matters
documents
audit_log


## Archive layout + audit chain

In [6]:
print("\n".join(arts["archive_bin"][:10]))
print("--- audit events ---")
al = arts["catalog"].get("audit_log") or {}
cols = al.get("columns", [])
for row in al.get("rows", [])[:8]:
    rec = dict(zip(cols, row))
    print(" ", rec.get("event_name"), "@", str(rec.get("created_at"))[:19])


LAB-MATTER/contract/papertrail.json
LAB-MATTER/contract/papertrail.txt
--- audit events ---
  None @ None
  None @ None
  None @ None
  None @ None


### Who eats what

- **conveyor stages** <- manifest `stage` (+ `updated_at`)
- **review siding queue** <- review-bin manifests
- **metrics panels** <- catalog rows grouped by `stage` / `doc_type`
- **trace links** <- `trace_id` (Langfuse) + `session_id = matter_id`

Same identifiers, four surfaces — that thread-through is the whole
integration contract.

## Bench teardown

In [7]:
lab.close_sandbox(env6)
print("sandbox closed")


sandbox closed


## Where to go next

- **07 · multi_document_matters** — many docs, one matter_id
- **08 · observability_traces** — the fifth surface (Langfuse)